# 02 · Compound Splitter (contest exercise)

**Style C: contest simulation**, a close copy of T25-G-WORDSEG ("Combinatorial Word Segmentation", IOAI 2025
GAITE): the same JSON format (`{word: [0/1 per char]}`), the same segment-F1 metric, and training that must happen
inside the notebook. **Traces to** `notes/ioai-task-patterns.md` P1, P2, P3, P9.

## 1. Problem Description

Compound words glue shorter words together ("Autobahnanschlussstelle" = Autobahn + Anschluss + Stelle). Here
compounds are built from English news vocabulary ("worldcupfinal"). Split them back into their words.

## 2. Dataset

JSON-style dicts `{compound: labels}`, where `labels[i] = 1` if a word ends at character `i` (so the last label is always 1).

| Split | Words | Labels |
|---|---|---|
| `train` | 60,000 | ✅ |
| `val` | 8,000 | empty lists (Leaderboard A) |
| `test` | 8,000 | empty lists (Leaderboard B, **final**) |

The validation and test compounds contain words that never appear inside any training compound, just as a German
test set contains words your training list lacks.

## 3. Task

Fill in the labels for every word in `val` and `test`.

## 4. Submission

`submission.zip` containing `submissionval.json` and `submissiontest.json`, in the same format as the training
data. A word whose label list is missing or has the wrong length scores 0.

## 5. Scoring

The T25-G-WORDSEG metric: for each word, the F1 between the predicted and true **segments** (a segment counts
only if both its boundaries are right), averaged over words.

- Baseline (Leaderboard B): **0.05** (the official per-character MLP with no context, below)
- Reference (Leaderboard B): **0.89**, which is your target

## 6. Requirements

- The notebook must include training and runs in **≤ 10 minutes on a T4** end to end (GAITE allowed 20 minutes).
- No pretrained models, no external word lists (the grader cannot see the internet).

In [ ]:
import time, os, json, zipfile, random, re, collections, base64, zlib
T0 = time.time()
TIME_LIMIT_MIN = 10
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

### Data loading (DO NOT CHANGE, DO NOT DECODE)

In [ ]:
from datasets import load_dataset
_texts = load_dataset("fancyzhx/ag_news", split="train")["text"][:40000]
_cnt = collections.Counter(w for t in _texts for w in re.findall(r"[a-z]+", t.lower()))
_WORDS = [w for w, _ in _cnt.most_common(6000) if 3 <= len(w) <= 8][:2500]
_BLOB = ("eNqFUttOwzAMfd9XWDylUKpuCDRV8CVVNYXFG2FtEiUpFUL8O3baAREg8lT5XHxsVw/O+ghmHNwryADGrRQeYLe3BIxGBTFZr0LRrICeN0d4IE7lpVF2qIgqxz7uqC429ea2SKykIF4gZ1QiYFxMZtihHwglTcWfY5RRWyN6NBktIBqitanW6g4O1oMGbZJB29BXFHW1hUv4Ju26JB7NP/Jfxc1ZHVBxfgo+Z+GVDPKEwpTgdrP5shJ+doxEf3v/LExPusfkTFAB92C+yPxOy/yUAo/og9iUsCzv/Jz0kZfYdlmZp9jxFHSBI4pTkRsnpbU9CZcV6EPqNB9McJbzAIB9wLTmnxbcvJLOoVGC/dosLQ/GVVp3HnqivhcX1bPVRGCPHKYsE2fn/TawtyZqM2JGYaiSSokpl/by8fdluHRS7tUk0hWx6o7uOqcs4BrWBVxBu861dJh26siTRJ+Axzh6w1gqRS81/0Tp9Hc1vRLqqp6Tvcj+DG0X5GZGIob4B7Q0SMYlW5SJvfoAwKb5YA==")
exec(zlib.decompress(base64.b64decode(_BLOB)).decode())
train, _val, _test = _compounds(_WORDS)
val = {w: [] for w in _val}
test = {w: [] for w in _test}
del _val, _test, _texts, _cnt                     # the organisers keep the answers
print(len(train), len(val), len(test)); print(list(train.items())[:3])

### The official metric (DO NOT CHANGE)

In [ ]:
def get_segments(labels):
    segs, start = [], 0
    for i, v in enumerate(labels):
        if v == 1: segs.append((start, i + 1)); start = i + 1
    if start < len(labels): segs.append((start, len(labels)))
    return segs

def average_f1(gold: dict, pred: dict) -> float:
    scores = []
    for w, g in gold.items():
        p = pred.get(w)
        if p is None or len(p) != len(g): scores.append(0.0); continue
        gs, ps = set(get_segments(g)), set(get_segments(p))
        tp = len(gs & ps)
        pr, rc = (tp / len(ps) if ps else 0), (tp / len(gs) if gs else 0)
        scores.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(scores))

### Shared data utilities

In [ ]:
chars = sorted({c for w in train for c in w})
char2idx = {c: i + 1 for i, c in enumerate(chars)}          # 0 = padding
VOCAB = len(char2idx) + 1

def collate(batch):
    L = max(len(w) for w, _ in batch)
    ids = torch.zeros(len(batch), L, dtype=torch.long); y = torch.zeros(len(batch), L)
    for i, (w, lab) in enumerate(batch):
        ids[i, :len(w)] = torch.tensor([char2idx.get(c, 0) for c in w])
        if lab: y[i, :len(lab)] = torch.tensor(lab, dtype=torch.float)
    return ids, y, torch.tensor([len(w) for w, _ in batch])

### Baseline (replace this): the official GAITE per-character MLP

Each character is classified from its own one-hot vector, with no context at all.

In [ ]:
class MyModel(nn.Module):
    def __init__(self, vocab, hidden=128):
        super().__init__()
        self.vocab = vocab
        self.fc1 = nn.Linear(vocab, hidden); self.fc2 = nn.Linear(hidden, hidden); self.out = nn.Linear(hidden, 1)
    def forward(self, ids, lengths):
        x = F.one_hot(ids, self.vocab).float()
        return self.out(F.relu(self.fc2(F.relu(self.fc1(x))))).squeeze(-1)

def train_model():
    model = MyModel(VOCAB).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(list(train.items()), batch_size=128, shuffle=True, collate_fn=collate)
    for epoch in range(3):
        for ids, y, lengths in loader:
            logits = model(ids.to(device), lengths)
            mask = torch.arange(ids.shape[1])[None] < lengths[:, None]
            loss = F.binary_cross_entropy_with_logits(logits[mask.to(device)], y.to(device)[mask.to(device)])
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"epoch {epoch} loss {loss.item():.4f}")
    return model

### Train, predict, validate locally

In [ ]:
model = train_model()

@torch.no_grad()
def predict(model, words):
    model.eval(); out = {}
    for i in range(0, len(words), 1024):
        chunk = words[i:i + 1024]
        ids, _, lengths = collate([(w, []) for w in chunk])
        pred = (model(ids.to(device), lengths) > 0).int().cpu()
        for j, w in enumerate(chunk):
            p = pred[j, :len(w)].tolist(); p[-1] = 1
            out[w] = p
    return out

held = dict(list(train.items())[-3000:])                   # seen during training: optimistic
print("F1 on 3k training words:", round(average_f1(held, predict(model, list(held))), 4))

### Submission

In [ ]:
sub_val, sub_test = predict(model, list(val)), predict(model, list(test))
with zipfile.ZipFile("submission.zip", "w") as z:
    z.writestr("submissionval.json", json.dumps(sub_val))
    z.writestr("submissiontest.json", json.dumps(sub_test))
print("wrote submission.zip")

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    _, gold_val, gold_test = _compounds(_WORDS)
    with zipfile.ZipFile(zip_path) as z:
        pv = json.loads(z.read("submissionval.json")); pt = json.loads(z.read("submissiontest.json"))
    return average_f1(gold_val, pv), average_f1(gold_test, pt)

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")
elapsed = time.time() - T0
print(f"total runtime {elapsed / 60:.1f} min (limit {TIME_LIMIT_MIN} min)")
assert elapsed < TIME_LIMIT_MIN * 60, "over the time limit: on the platform this run would score 0"